# 08 – Density-based: DBSCAN
Parameter selection is the difficulty: eps from the k-distance elbow or a quantile, min_samples fixed. We record degenerate behaviour (almost everything noise / no cluster).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
sys.path.insert(0, str(root))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    from IPython.display import display, Image
except ImportError:
    display = print
PARSED = root / "data" / "processed" / "gas_sensor_drift_parsed.csv"
USE_STANDIN = False   # True ONLY to test the notebook without UCI data (results then are NOT valid)
if USE_STANDIN:
    from src.synthetic import make_standin
    df = make_standin(5000)
    print("!! STAND-IN DATA - not UCI")
else:
    assert PARSED.exists(), "Run notebook 01 (or `python -m src.data_loader`) first"
    df = pd.read_csv(PARSED)
FIG = root / "results" / "figures"; TAB = root / "results" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
print(df.shape)

In [ ]:
from src.experiments import Condition, prepare_dataset
from src.evaluation import evaluate_scores
from src.registry import make_detector
cond = Condition(anomaly_type="point", n_dims=64, ratio=0.05, noise=0.05, n_samples=2000, seed=0)
data = prepare_dataset(df, cond)
X, y = data["X"], data["y"]
print("baseline set:", X.shape, "| anomalies:", int(y.sum()))
def run(name, params=None, seed=0):
    det = make_detector(name, params, seed); s = det.fit_score(X)
    m = evaluate_scores(y, s, cond.ratio); m.update({"algorithm": name, **{k: v for k, v in det.extras.items() if not isinstance(v, dict)}})
    return m, s

In [ ]:
from sklearn.neighbors import NearestNeighbors
kd = np.sort(NearestNeighbors(n_neighbors=10).fit(X).kneighbors(None)[0][:, -1])
plt.plot(kd); plt.xlabel('points sorted by 10-NN distance'); plt.ylabel('10-NN distance'); plt.title('k-distance graph (DBSCAN eps selection)'); plt.savefig(FIG/'nb08_kdistance.png', dpi=200, bbox_inches='tight')

In [ ]:
rows=[]
for strat,q in [('knee',None),('quantile',0.8),('quantile',0.9),('quantile',0.95)]:
    p={'eps_strategy':strat}; 
    if q: p['quantile']=q
    m,_=run('DBSCAN', p); m.update(strategy=strat, q=q); rows.append(m)
display(pd.DataFrame(rows)[['strategy','q','roc_auc','pr_auc','eps','n_clusters','noise_fraction','degenerate']])

### Result log (fill in after running)
*Not yet executed.* Copy the numbers printed above into your report; do not quote numbers from memory.